# Project (Section 7)

This is where we left off in the previous section project:

In [7]:
from datetime import date
from enum import Enum
from uuid import uuid4
from pydantic import BaseModel, ConfigDict, Field, field_serializer
from pydantic.alias_generators import to_camel
from pydantic import UUID4


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


Create an annotated type, named `BoundedString` to define a string that has a minimum of 2 characters, and no more than 50 characters.

Modify your `Automobile` model to use this annotated type for the following fields:
- `manufacturer`
- `series_name`
- `vin`
- `registration_country`
- `license_plate`

Create an annotated type, called `BoundedList` that uses a type variable to define a list of elements with a minimum of `1` element and a maximum of `5` elements.

Using this annotated type, add a new field to the model as follows:
- field name should be `top_features`
- place it just before the `vin` field
- it should both deserialize from and serialize to `topFeatures`
- if should be a bounded list of strings, which themselves should be bounded to a minimum of `2` chars, and no more than `50`. (Hint: use the `BoundedString` type you create as the type when you define the field type in your model with `BoundedList`)
- make it optional, with a default of `None`

Use this data to test your model:

In [8]:
from typing import Annotated
from pydantic import Field

In [9]:
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[BoundedString], Field(min_length=2, max_length=5)]

In [10]:
class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel, # 
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4) 
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD", 
        serialization_alias="baseMSRPUSD"
    )
    top_features: BoundedList | None = None # Field(alias="topFeatures", default=None) is not needed due to the macro setting
    vin: BoundedString
    number_of_doors: int = Field(
        default=4, 
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
    )
    registration_country: BoundedString | None = None
    license_plate: BoundedString | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

In [11]:
from uuid import UUID

data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [12]:
car = Automobile.model_validate(data)

In [13]:
assert car.model_dump(by_alias=True) == expected_serialized_by_alias

In [14]:
import pytest
from pydantic import ValidationError

In [15]:
data_false = data.copy()
data_false["registrationCountry"]="F"

In [16]:
data_false

{'id': 'c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7',
 'manufacturer': 'BMW',
 'seriesName': 'M4 Competition xDrive',
 'type': 'Convertible',
 'isElectric': False,
 'completionDate': '2023-01-01',
 'msrpUSD': 93300,
 'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
 'vin': '1234567890',
 'doors': 2,
 'registrationCountry': 'F',
 'licensePlate': 'AAA-BBB'}

In [17]:
with pytest.raises(ValidationError, match=r"at least 2 characters"):
    Automobile.model_validate(data_false)

In [18]:
data_false2 = data.copy()
data_false2["topFeatures"]=["feat1"]

In [20]:
with pytest.raises(ValidationError, match='List should have at least 2 items after validation'):
    Automobile.model_validate(data_false2)

You should also test that you get validation errors if you attempt to deserialize data that does not conform to the constraints. 

> Suggestion: create simple single field models to just test your Annotated types and make sure they work as expected. Once you have done that, go ahead and use them in your main model.